# NeMo Guardrails + LangGraph: Intent-Based Tool Filtering

In [ ]:
# Environment setup
import os
import nest_asyncio
from dotenv import load_dotenv

nest_asyncio.apply()
load_dotenv(dotenv_path='../.env')

os.environ["OPENAI_API_KEY"] = os.environ['UNIFIED_LLM_KEY']
os.environ["OPENAI_API_BASE"] = os.environ['BASE_URL']
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "agent_guardrails"
os.environ["LANGCHAIN_API_KEY"] = os.environ.get('LANGSMITH_API_KEY', '')

In [ ]:
from nemoguardrails import RailsConfig
from nemoguardrails.integrations.langchain.runnable_rails import RunnableRails

## Guardrails Config (Intent Detection)

In [ ]:
config_dir = "config/guardrails_intent"
os.makedirs(config_dir, exist_ok=True)

# config.yml - FIXED: embeddings_only: false for reliable intent detection
with open(f"{config_dir}/config.yml", 'w') as f:
    f.write("""
models:
  - type: main
    engine: openai
    model: gpt-4o

embeddings:
  engine: fastembed
  model: sentence-transformers/all-MiniLM-L6-v2

rails:
  dialog:
    user_messages:
      embeddings_only: false  # Use both LLM and embeddings (more reliable)

logging:
  internal_events: true
""")

# rails.co - Intent definitions (with MORE examples for better matching)
with open(f"{config_dir}/rails.co", 'w') as f:
    f.write("""define user search books
  "find me a book about Python"
  "find me books about Python programming"
  "search for mystery novels"
  "show me programming books"
  "I'm looking for books on machine learning"
  "do you have any books about Python"
  "can you recommend Python books"
  "looking for AI books"
  "find books on data science"

define user ask about authors
  "who wrote The Great Gatsby?"
  "tell me about Stephen King"
  "what books did J.K. Rowling write"
  "who is the author of Harry Potter"

define user check inventory
  "is this book in stock?"
  "do you have Harry Potter available?"
  "check availability of Python Crash Course"
  "how many copies do you have"

define user general conversation
  "hello"
  "what can you do?"
  "help me"
""")

print("✅ Config created")

## Debug: Test Intent Detection

In [ ]:
# Load and test guardrails directly
test_config = RailsConfig.from_path(config_dir)
test_guardrails = RunnableRails(config=test_config, passthrough=False, verbose=False)

test_input = "Find me books about Python programming"
print(f"Testing: {test_input}\n")

response = test_guardrails.invoke(
    {"input": test_input},
    config={"internal_events": True}
)

# Check internal events
events = response.get("log", {}).get("internal_events", [])
print(f"Total events: {len(events)}\n")

# Find UserIntent events
user_intents = [e for e in events if e.get("type") == "UserIntent"]
if user_intents:
    print("✅ UserIntent events found:")
    for intent_event in user_intents:
        print(f"   Intent: {intent_event.get('intent')}")
else:
    print("❌ No UserIntent events found!")
    print("\nAll events:")
    for i, e in enumerate(events[:5]):  # Show first 5
        print(f"   {i+1}. {e.get('type')}: {e}")

## Tools Definition

In [ ]:
from langchain_core.tools import tool

@tool
def search_books(query: str) -> str:
    """Search for books by title or topic."""
    books_db = {"python": ["Python Crash Course", "Automate the Boring Stuff"], "ai": ["AI: A Modern Approach", "Deep Learning"]}
    for key, books in books_db.items():
        if key in query.lower():
            return f"Found: {', '.join(books)}"
    return "No books found."

@tool
def search_by_genre(genre: str) -> str:
    """Search books by genre."""
    genres = {"fiction": ["The Great Gatsby", "1984"], "sci-fi": ["Dune", "Foundation"]}
    return f"Books in {genre}: {', '.join(genres.get(genre.lower(), []))}" if genre.lower() in genres else f"No books in {genre}"

@tool
def get_author_info(author_name: str) -> str:
    """Get author information."""
    authors = {"stephen king": "American horror author", "j.k. rowling": "Creator of Harry Potter"}
    for author, info in authors.items():
        if author in author_name.lower():
            return f"{author.title()}: {info}"
    return f"No info for {author_name}"

@tool
def get_author_books(author_name: str) -> str:
    """List books by author."""
    author_books = {"stephen king": ["The Shining", "IT"], "j.k. rowling": ["Harry Potter Series"]}
    for author, books in author_books.items():
        if author in author_name.lower():
            return f"{author.title()}: {', '.join(books)}"
    return f"No books for {author_name}"

@tool
def check_book_availability(book_title: str) -> str:
    """Check book stock."""
    inventory = {"python crash course": 15, "harry potter": 8}
    for book, qty in inventory.items():
        if book in book_title.lower():
            return f"✅ '{book.title()}' - {qty} in stock"
    return f"Not found: {book_title}"

ALL_TOOLS = [search_books, search_by_genre, get_author_info, get_author_books, check_book_availability]

## Intent → Tool Mapping

In [ ]:
INTENT_TOOL_MAP = {
    "search books": [search_books, search_by_genre],
    "ask about authors": [get_author_info, get_author_books],
    "check inventory": [check_book_availability],
    "general conversation": [],
    "unknown": ALL_TOOLS,
}

## LangGraph Setup

In [ ]:
from typing import Annotated, Any
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode

# State
class IntentState(TypedDict):
    messages: Annotated[list[Any], add_messages]
    intent: str

# Extract intent from guardrails - IMPROVED VERSION
def extract_intent(response: dict) -> str:
    """Extract intent from NeMo Guardrails response."""
    try:
        log = response.get("log", {})
        events = log.get("internal_events", [])

        # Search for UserIntent events (most recent first)
        for event in reversed(events):
            if event.get("type") == "UserIntent":
                intent = event.get("intent", "general")
                # Normalize: strip whitespace and convert to lowercase
                intent = intent.strip().lower().replace("_", " ")
                return intent

        # No UserIntent found
        print("⚠️  No UserIntent events found - using 'general'")
        return "general"

    except Exception as e:
        print(f"⚠️  Intent extraction error: {e}")
        return "general"

# Chatbot with intent-based tool filtering
def create_chatbot():
    llm = ChatOpenAI(model="gpt-4o", temperature=0)
    config = RailsConfig.from_path(config_dir)
    guardrails = RunnableRails(config=config, passthrough=False, verbose=False)
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful bookstore assistant."),
        ("placeholder", "{messages}"),
    ])

    def chatbot(state: IntentState) -> dict:
        # Get user message content
        last_msg = state["messages"][-1]
        user_input = last_msg.content if hasattr(last_msg, 'content') else str(last_msg)

        # Detect intent
        gr_response = guardrails.invoke(
            {"input": user_input},
            config={"internal_events": True}
        )
        intent = extract_intent(gr_response)
        print(f"🎯 Intent: {intent} | Tools: {len(INTENT_TOOL_MAP.get(intent, ALL_TOOLS))}/{len(ALL_TOOLS)}")

        # Filter tools & invoke LLM
        tools = INTENT_TOOL_MAP.get(intent, ALL_TOOLS)
        llm_with_tools = llm.bind_tools(tools) if tools else llm
        response = (prompt | llm_with_tools).invoke(state)

        return {"messages": [response], "intent": intent}

    return chatbot

# Build graph
def route_tools(state: IntentState) -> str:
    return "tools" if hasattr(state["messages"][-1], 'tool_calls') and state["messages"][-1].tool_calls else END

graph = StateGraph(IntentState)
graph.add_node("chatbot", create_chatbot())
graph.add_node("tools", ToolNode(ALL_TOOLS))
graph.add_edge(START, "chatbot")
graph.add_conditional_edges("chatbot", route_tools, {"tools": "tools", END: END})
graph.add_edge("tools", "chatbot")

app = graph.compile()
print("✅ Graph ready")

## Test

In [ ]:
# Book search
r = app.invoke({"messages": [{"role": "user", "content": "Find me books about Python programming"}], "intent": "unknown"})
print(f"\nDetected: {r['intent']}")
print(f"Response: {r['messages'][-1].content}")

In [ ]:
# Author info
r = app.invoke({"messages": [{"role": "user", "content": "Tell me about Stephen King"}], "intent": "unknown"})
print(f"\nDetected: {r['intent']}")
print(f"Response: {r['messages'][-1].content}")

In [ ]:
# Inventory
r = app.invoke({"messages": [{"role": "user", "content": "Is Harry Potter in stock?"}], "intent": "unknown"})
print(f"\nDetected: {r['intent']}")
print(f"Response: {r['messages'][-1].content}")